# 6주차 추가 실습 · PyTorch로 신경망 학습하기

**이번 실습:** 공개 이미지 데이터 CIFAR-10으로 사진을 10가지로 분류하는 CNN을 학습하며 신경망 학습의 내부 순서를 직접 봅니다. 기존 `06주차_신경망_검증하기.ipynb`는 분류·검증 실습이고, 이 파일은 `class`, 자동 미분, 학습 반복문의 역할을 배우는 별도 실습입니다.

기본 구역 1~9를 위에서부터 실행합니다. RNN은 맨 아래 선택 구역입니다. **이 실습은 Colab에서 진행합니다.** 실행 전에 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택합니다. 저장된 출력은 Colab T4에서 실제 실행한 예시이며 소수점 값은 환경에 따라 조금 달라질 수 있습니다.

VS Code에서도 CPU로 실행됩니다. 커널은 **Python (semi-ai)**를 고르고, 터미널에서 `conda activate semi-ai` 후 `pip install torchvision==0.15.2`를 한 번 실행합니다. CPU에서는 한 바퀴에 1분 정도 걸리므로 1바퀴만 학습합니다.

## 0. 실습 폴더 준비

VS Code와 Colab 어디서 열어도 `data` 폴더를 찾도록 실습 폴더 위치를 정합니다. Colab에서는 처음 한 번 실습 자료를 내려받습니다.

In [ ]:
import sys, subprocess  # 실행 환경 확인과 명령 실행 도구입니다.
from pathlib import Path  # 파일 위치 도구입니다.
if "google.colab" in sys.modules:  # Colab에서 실행 중이라면
    folder = Path("/content/semiconductor_ai_beginner")  # 실습 자료를 둘 위치입니다.
    if not folder.exists():  # 아직 내려받지 않았다면
        subprocess.run(["git", "clone", "-q", "https://github.com/SeokJH-TAs/semiconductor_ai_beginner.git", str(folder)], check=True)  # 실습 자료 전체를 내려받습니다.
else:  # VS Code 등 내 컴퓨터에서 실행 중이라면
    folder = Path.cwd()  # 현재 폴더입니다.
    if folder.name == "notebooks":  # 노트북 폴더 안이라면
        folder = folder.parent  # 실습 전체 폴더로 올라갑니다.
print(folder)  # 실습 폴더 위치를 확인합니다.

## 1. 도구와 사진 불러오기

`torch.Tensor`는 PyTorch가 계산하는 숫자 묶음입니다. `nn`은 신경망 층과 손실 함수를 모아 둔 도구입니다. `torchvision`은 공개 이미지 데이터를 내려받는 도구입니다. 이번 데이터는 **32×32 컬러 사진 6만 장**(학습 5만, 시험 1만)이며 비행기·자동차·새·고양이·사슴·개·개구리·말·배·트럭 10가지입니다.

`device`는 계산할 장치입니다. GPU가 있으면 `cuda`, 없으면 `cpu`입니다. 뒤에서 모델과 배치를 `.to(device)`로 같은 장치에 올립니다. 원본 서버가 느릴 때가 있어 같은 파일을 수업 저장소에 올려 두고 거기서 받습니다.

In [ ]:
import time  # 시간을 재는 도구입니다.
import numpy as np  # 배열 계산 도구입니다.
import matplotlib.pyplot as plt  # 그림 도구입니다.
plt.rcParams["font.size"] = 14  # 발표 화면에서도 축 글자를 읽기 쉽게 키웁니다.
import torch  # 텐서와 자동 미분 도구입니다.
from torch import nn  # 신경망 도구를 꺼냅니다.
from torch.utils.data import DataLoader  # 배달 도구입니다.
from torchvision import datasets, transforms  # 공개 이미지 데이터와 변환 도구입니다.
device = "cuda" if torch.cuda.is_available() else "cpu"  # GPU가 있으면 cuda, 없으면 cpu입니다.
datasets.CIFAR10.url = "https://github.com/SeokJH-TAs/semiconductor_ai_beginner/releases/download/data/cifar-10-python.tar.gz"  # 수업 저장소의 같은 파일을 받습니다.
to_tensor = transforms.ToTensor()  # 사진을 0~1 범위의 텐서로 바꾸는 변환입니다.
train_data = datasets.CIFAR10(root=folder / "data" / "cifar10", train=True, download=True, transform=to_tensor)  # 학습용 5만 장입니다.
test_data = datasets.CIFAR10(root=folder / "data" / "cifar10", train=False, download=True, transform=to_tensor)  # 시험용 1만 장입니다.
class_names = train_data.classes  # 10가지 이름 목록입니다.
print("PyTorch:", torch.__version__, "장치:", device)  # 실제 실행 버전과 장치를 봅니다.
print("학습 장수:", len(train_data), "시험 장수:", len(test_data))  # 50000, 10000입니다.
print("종류:", class_names)  # 10가지 이름입니다.

## 2. 사진을 텐서로 보기

`ToTensor()`가 사진 한 장을 `(채널, 높이, 너비)` 모양의 `float32` 텐서로 바꿉니다. 채널 3은 빨강·초록·파랑 판 3장이고, 한 장은 3×32×32 = 3,072개 숫자입니다. 3주차 공정 표의 한 행은 숫자 3개였습니다. 정답은 0~9 번호 하나이며 `class_names`로 이름을 찾습니다.

`permute(1, 2, 0)`은 그림 도구가 원하는 `(높이, 너비, 채널)` 순서로 축을 바꿉니다.

In [ ]:
image, label = train_data[0]  # 첫 사진과 정답 번호입니다.
print("한 장의 모양:", image.shape, "숫자 개수:", image.numel())  # (3, 32, 32), 3072입니다.
print("정답 번호:", label, "이름:", class_names[label])  # 번호와 이름입니다.
print(image[0, :2, :5])  # 빨강 판의 왼쪽 위 2×5 칸 숫자입니다.
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))  # 사진 6장을 나란히 놓습니다.
for i, axis in enumerate(axes):  # 자리마다 하나씩
    image, label = train_data[i]  # i번째 사진과 정답입니다.
    axis.imshow(image.permute(1, 2, 0))  # 축 순서를 바꿔 사진으로 그립니다.
    axis.set_title(class_names[label])  # 정답 이름을 제목으로 붙입니다.
    axis.axis("off")  # 눈금은 숨깁니다.
plt.tight_layout()  # 여백을 맞춥니다.
plt.show()  # 사진을 보여 줍니다.

## 3. Dataset과 DataLoader

`train_data`는 사진과 정답을 짝지어 보관하는 Dataset입니다. `DataLoader`는 그 짝을 작은 묶음 **배치**로 꺼내 줍니다. `batch_size=128`은 한 번에 최대 128장, `shuffle=True`는 학습 순서를 섞는다는 뜻입니다. 5만 장이므로 한 바퀴에 391배치이며 마지막은 80장입니다.

`next(iter(loader))`는 배달을 시작해서 첫 묶음만 꺼내는 표현입니다. 반복 학습에서는 `for batch_x, batch_y in loader`를 사용합니다. `num_workers=0`으로 별도 작업 프로세스를 만들지 않아 노트북에서도 단순하게 실행합니다. 시험용 loader는 채점에만 쓰므로 섞지 않습니다.

In [ ]:
torch.manual_seed(42)  # 순서와 시작값을 재현하기 위한 씨앗입니다.
loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=0)  # 학습 묶음을 꺼냅니다.
test_loader = DataLoader(test_data, batch_size=256, shuffle=False, num_workers=0)  # 시험 묶음입니다. 섞지 않습니다.
batch_x, batch_y = next(iter(loader))  # 첫 배치 하나를 꺼냅니다.
print("사진 수:", len(train_data), "배치 수:", len(loader))  # 50000과 391입니다.
print("배치 모양:", batch_x.shape, batch_y.shape)  # (128, 3, 32, 32), (128,)입니다.
print("첫 정답:", batch_y[0].item(), class_names[batch_y[0]])  # 이 배치 첫 사진의 정답입니다.

## 4. class와 nn.Module

`class`는 같은 구조의 모델을 만드는 설계도입니다. `SmallCNN()`을 실행하면 실제 모델 하나가 생깁니다. 괄호 안의 `nn.Module`을 물려받아 층·가중치를 등록하고 `parameters()`, `train()`, `eval()` 기능을 사용합니다.

`self`는 지금 만든 모델 자신입니다. `__init__`은 모델을 처음 만들 때 층을 준비하고, `super().__init__()`은 부모인 nn.Module을 준비합니다. `forward`에는 입력에서 예측까지의 계산만 적습니다. `network(batch_x)`가 이 계산을 호출합니다. 역전파는 forward 안에 쓰지 않습니다.

사진은 옆 픽셀과의 관계가 중요하므로 숫자 전체를 한 번에 섞는 `Linear` 대신 **3×3 창으로 옆끼리만 보는 `Conv2d`**를 씁니다. `Conv2d(3, 32)`는 판 3장(RGB)을 받아 무늬 32종류의 지도를 만들고, `MaxPool2d(2)`는 2×2마다 가장 큰 값만 남겨 크기를 반으로 줄입니다. 입력 `(배치, 3, 32, 32)`를 `(배치, 32, 16, 16)`, `(배치, 64, 8, 8)`로 바꾼 뒤 한 줄로 펴서 `(배치, 10)` 점수로 만듭니다. 분류 출력층에는 sigmoid를 붙이지 않고 손실 함수가 점수를 확률로 바꿉니다.

In [ ]:
class SmallCNN(nn.Module):  # nn.Module 기능을 가진 새 모델 설계도입니다.
    def __init__(self):  # 모델을 만들 때 한 번 실행합니다.
        super().__init__()  # nn.Module의 기본 준비를 합니다.
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)  # 판 3장을 무늬 지도 32장으로 바꿉니다.
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 지도 32장을 64장으로 바꿉니다.
        self.activation = nn.ReLU()  # 중간 숫자에 비선형 변환을 합니다. 음수는 0이 됩니다.
        self.pool = nn.MaxPool2d(2)  # 가로·세로를 반으로 줄입니다.
        self.flatten = nn.Flatten()  # 격자를 한 줄로 폅니다.
        self.output = nn.Linear(64 * 8 * 8, 10)  # 숫자 4096개로 점수 10개를 만듭니다.

    def forward(self, x):  # 모델에 입력을 넣으면 실행할 계산입니다.
        hidden = self.pool(self.activation(self.conv1(x)))  # 첫 블록. (3,32,32) → (32,16,16)
        hidden = self.pool(self.activation(self.conv2(hidden)))  # 둘째 블록. (32,16,16) → (64,8,8)
        flat = self.flatten(hidden)  # 한 줄로 폅니다. 4096개
        prediction = self.output(flat)  # 마지막 층 계산입니다. 점수 10개
        return prediction  # 예측을 호출한 곳으로 돌려줍니다.

In [ ]:
torch.manual_seed(42)  # 모델 시작값을 고정합니다.
demo = SmallCNN()  # 설계도로 실제 모델 하나를 만듭니다.
hidden = demo.pool(demo.activation(demo.conv1(batch_x)))  # 첫 블록 출력을 꺼냅니다.
demo_prediction = demo(batch_x)  # forward를 포함한 전체 계산입니다.
print(demo)  # 등록된 층을 봅니다.
print("모양:", batch_x.shape, hidden.shape, demo_prediction.shape)  # (128,3,32,32), (128,32,16,16), (128,10)입니다.
print("첫 예측:", demo_prediction[0].detach().numpy().round(2))  # 미분 연결을 떼고 숫자만 읽습니다. 아직 학습 전입니다.
for name, parameter in demo.named_parameters():  # 등록된 가중치·편향을 하나씩 꺼냅니다.
    print(name, parameter.shape)  # optimizer가 바꿀 숫자들의 이름과 모양입니다.
print("학습할 숫자 개수:", sum(p.numel() for p in demo.parameters()))  # 전부 더한 개수입니다.

## 5. 한 배치에서 backward와 optimizer 보기

5주차에서는 `dw`를 직접 구했습니다. PyTorch에서는 `loss.backward()`가 연결된 계산을 따라가 가중치의 `.grad`에 기울기를 채웁니다. 이 줄만으로 가중치를 바꾸지는 않습니다.

`optimizer`는 등록된 파라미터를 어떤 규칙으로 얼마나 움직일지 정합니다. 여기서는 5주차와 연결되는 SGD를 사용합니다. `step()`이 실제 값을 갱신합니다. `zero_grad(set_to_none=True)`는 이전 기울기 기록을 비웁니다. 생략하면 기울기가 누적됩니다. 초기화 뒤 `.grad`가 `None`인 것은 정상입니다.

손실은 분류용 `CrossEntropyLoss`입니다. 점수 10개를 확률로 바꿔 정답 번호에 준 확률이 작을수록 큰 값이 됩니다. 10가지 중 찍으면 약 2.3(= ln 10)입니다. 모델과 배치를 `.to(device)`로 같은 장치에 올려야 계산됩니다. 아래 `demo`는 한 번만 갱신해 보는 관찰용 모델입니다. 6번에서는 새 모델로 전체 학습을 시작합니다.

In [ ]:
loss_function = nn.CrossEntropyLoss()  # 점수 10개와 정답 번호를 비교하는 분류 손실입니다.
demo = demo.to(device)  # 관찰용 모델을 장치에 올립니다.
optimizer = torch.optim.SGD(demo.parameters(), lr=0.05, momentum=0.9)  # demo의 숫자를 보폭 0.05로 바꿀 도구입니다.
optimizer.zero_grad(set_to_none=True)  # 지난 기울기를 비웁니다.
print("비운 기울기:", demo.conv1.weight.grad)  # 아직 기울기가 없어 None입니다.
prediction = demo(batch_x.to(device))  # 현재 가중치로 예측합니다. 배치도 같은 장치에 올립니다.
loss = loss_function(prediction, batch_y.to(device))  # 정답과 비교한 손실입니다.
print("배치 손실:", loss.item())  # 텐서 속 숫자 하나를 꺼냅니다.
print("갱신 전:", demo.conv1.weight[0, 0, 0].detach().clone().cpu())  # 첫 필터 첫 줄의 가중치 사본입니다.
loss.backward()  # 각 파라미터의 기울기를 계산합니다.
print("계산한 기울기:", demo.conv1.weight.grad[0, 0, 0].cpu())  # 같은 위치의 기울기입니다.
optimizer.step()  # 기울기를 사용해 가중치·편향을 바꿉니다.
print("갱신 후:", demo.conv1.weight[0, 0, 0].detach().cpu())  # 바뀐 가중치를 봅니다.

## 6. main 함수에서 전체 학습하기

`def main():`은 실행 순서를 묶은 함수입니다. 이름 자체에 특별한 학습 기능은 없습니다. 아래에서 `main()`을 호출해야 실행됩니다. 노트북에서는 셀을 순서대로 실행하므로 별도 실행 보호문 없이 호출합니다.

**epoch**는 학습 사진을 한 바퀴 도는 횟수입니다. 바깥 반복은 GPU에서 5바퀴(CPU에서는 1바퀴), 안쪽 반복은 배치를 하나씩 꺼냅니다. 배치마다 기울기 비우기, 예측, 손실, 역전파, 갱신을 수행합니다. 여기서는 SGD를 여러 번 반복합니다.

기록 손실은 각 배치의 갱신 직전 손실에 장수를 곱해 합친 뒤 5만으로 나눈 값입니다. 마지막 배치가 작아도 장마다 같은 비중으로 셉니다. 시험 사진은 학습에 사용하지 않습니다. 바퀴마다 걸린 시간을 함께 적습니다. 아래 함수가 앞 구역의 `loader` 등을 사용하므로 1~5번을 먼저 실행해야 합니다.

`accuracy`는 시험 사진 1만 장의 맞힌 비율입니다. 점수 10개 중 가장 큰 번호(`argmax`)가 예측이고 정답과 같으면 맞힌 것입니다. 학습 전·후를 비교하려고 main 안에서 학습 전에 한 번 먼저 잽니다.

In [ ]:
def accuracy(network):  # 시험 사진 1만 장에서 맞힌 비율을 구합니다.
    correct = 0  # 맞힌 장수입니다.
    with torch.no_grad():  # 채점에는 미분 기록이 필요 없습니다.
        for test_x, test_y in test_loader:  # 시험 배치를 하나씩 받습니다.
            predicted = network(test_x.to(device)).argmax(dim=1)  # 점수가 가장 큰 번호가 예측입니다.
            correct = correct + (predicted == test_y.to(device)).sum().item()  # 정답과 같은 장수를 더합니다.
    return correct / len(test_data)  # 맞힌 비율입니다.

def main():  # 학습 전체의 실행 순서를 묶습니다.
    torch.manual_seed(42)  # 같은 시작 가중치에서 출발합니다.
    network = SmallCNN().to(device)  # 관찰용 demo와 별개인 새 모델입니다. 장치에 올립니다.
    loss_function = nn.CrossEntropyLoss()  # 분류 손실을 사용합니다.
    optimizer = torch.optim.SGD(network.parameters(), lr=0.05, momentum=0.9)  # 바꿀 숫자와 보폭을 정합니다.
    history = []  # 한 바퀴마다 손실을 저장합니다.
    before_accuracy = accuracy(network)  # 학습 전 시험 정확도입니다. 찍으면 약 0.1입니다.
    network.train()  # 학습 모드입니다. 이것만으로 학습하지는 않습니다.
    epochs = 5 if device == "cuda" else 1  # GPU는 5바퀴, CPU는 시간이 걸려 1바퀴만 돕니다.
    for epoch in range(epochs):  # 전체 학습 사진을 정한 바퀴만큼 봅니다.
        start = time.time()  # 이번 바퀴의 시작 시각입니다.
        total_loss = 0.0  # 이번 바퀴의 손실 합을 0에서 시작합니다.
        for batch_x, batch_y in loader:  # 입력·정답 배치를 하나씩 받습니다.
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)  # 배치를 모델과 같은 장치에 올립니다.
            optimizer.zero_grad(set_to_none=True)  # 이전 배치의 기울기를 비웁니다.
            prediction = network(batch_x)  # 순전파로 예측합니다.
            loss = loss_function(prediction, batch_y)  # 정답과 비교합니다.
            loss.backward()  # 기울기를 자동으로 계산합니다.
            optimizer.step()  # 가중치와 편향을 갱신합니다.
            total_loss = total_loss + loss.item() * len(batch_x)  # 장수를 고려해 더합니다.
        epoch_loss = total_loss / len(train_data)  # 한 장당 평균 손실입니다.
        history.append(epoch_loss)  # 그래프로 볼 기록을 남깁니다.
        print("epoch:", epoch + 1, "loss:", round(epoch_loss, 4), "시간:", round(time.time() - start, 1), "초")  # 진행 상황과 걸린 시간입니다.
    return network, history, before_accuracy  # 결과를 함수 밖으로 꺼냅니다.

In [ ]:
network, history, before_accuracy = main()  # 위에서 정의한 학습 함수를 실제로 실행합니다.
print("저장한 손실 수:", len(history))  # 바퀴 수와 같은지 확인합니다.

### CPU와 GPU 시간 비교

같은 학습을 CPU에서 20배치만 돌려 한 배치에 걸리는 시간을 재고, 391을 곱해 한 바퀴 시간을 예상합니다. GPU가 있으면 같은 방법으로 GPU도 재서 나란히 봅니다. 사진 한 장이 숫자 3,072개이고 5만 장을 여러 바퀴 돌리면 이 차이가 수업 시간 전체를 좌우합니다.

In [ ]:
for name in ["cpu", "cuda"] if device == "cuda" else ["cpu"]:  # GPU가 있으면 둘 다, 없으면 CPU만 잽니다.
    timing_network = SmallCNN().to(name)  # 재기용 새 모델을 해당 장치에 올립니다.
    timing_optimizer = torch.optim.SGD(timing_network.parameters(), lr=0.05, momentum=0.9)  # 같은 갱신 도구입니다.
    start = time.time()  # 시작 시각입니다.
    for i, (batch_x, batch_y) in enumerate(loader):  # 배치를 하나씩 꺼냅니다.
        timing_optimizer.zero_grad(set_to_none=True)  # 기울기 비우기
        loss = loss_function(timing_network(batch_x.to(name)), batch_y.to(name))  # 예측과 손실
        loss.backward()  # 역전파
        timing_optimizer.step()  # 갱신
        if i == 19:  # 20배치만 재고
            break  # 멈춥니다.
    per_batch = (time.time() - start) / 20  # 한 배치에 걸린 시간입니다.
    print(name, "배치당", round(per_batch * 1000), "ms → 한 바퀴(391배치) 예상", round(per_batch * 391), "초")  # 장치별 예상 시간입니다.

## 7. 손실 곡선

손실이 줄어드는지 봅니다. 세로축은 학습 사진의 교차 엔트로피입니다. 학습 손실이 작다는 사실만으로 새로운 사진에도 잘 맞는다고 결론내리지 않습니다. CPU에서 1바퀴만 돌렸다면 점 하나만 찍힙니다.

In [ ]:
plt.figure(figsize=(6, 4))  # 손실 그림입니다.
plt.plot(range(1, len(history) + 1), history, marker="o")  # epoch별 손실 기록입니다.
plt.xlabel("Epoch")  # 전체 사진을 본 횟수입니다.
plt.ylabel("Training loss (cross entropy)")  # 학습 사진의 교차 엔트로피입니다.
plt.title("Learning curve")  # 학습 과정입니다.
plt.tight_layout()  # 여백을 맞춥니다.
plt.show()  # 그림을 보여 줍니다.

## 8. 학습 전·후 시험 정확도

`eval()`은 평가 모드로 바꾸고, `no_grad()`는 자동 미분 기록을 끕니다. 둘의 역할은 다릅니다. 이 모델의 Conv2d·ReLU는 모드 전환에 따라 계산이 달라지지 않지만, Dropout 등을 추가할 때도 같은 습관을 사용합니다.

시험 정답은 학습이 끝난 뒤 결과를 한 번 확인하는 데 사용합니다. 이 점수로 보폭·epoch를 계속 고르려면 기존 6주차처럼 별도의 검증 자료를 만들어야 합니다. 시험 사진 8장에 예측(정답)을 적어 봅니다. 틀린 것은 빨간 제목입니다.

In [ ]:
network.eval()  # 평가 모드로 바꿉니다.
after_accuracy = accuracy(network)  # 학습 후 시험 정확도입니다.
print("학습 전 정확도:", round(before_accuracy, 3))  # 학습 전입니다. 10가지 중 찍은 수준입니다.
print("학습 후 정확도:", round(after_accuracy, 3))  # 학습 후입니다.
test_x, test_y = next(iter(test_loader))  # 시험 첫 배치입니다.
with torch.no_grad():  # 평가에는 기울기를 기록하지 않습니다.
    predicted = network(test_x.to(device)).argmax(dim=1).cpu()  # 예측 번호입니다.
fig, axes = plt.subplots(1, 8, figsize=(14, 2.5))  # 사진 8장 자리입니다.
for i, axis in enumerate(axes):  # 자리마다
    axis.imshow(test_x[i].permute(1, 2, 0))  # 사진을 그립니다.
    color = "black" if predicted[i] == test_y[i] else "red"  # 맞으면 검정, 틀리면 빨강입니다.
    axis.set_title(f"{class_names[predicted[i]]}\n({class_names[test_y[i]]})", color=color, fontsize=10)  # 예측(정답)입니다.
    axis.axis("off")  # 눈금은 숨깁니다.
plt.tight_layout()  # 여백을 맞춥니다.
plt.show()  # 예측 결과를 보여 줍니다.

## 9. 역할을 말로 정리하기

| 코드 | 맡은 일 |
|---|---|
| datasets.CIFAR10 / DataLoader | 사진·정답 짝을 보관 / 배치로 꺼내기 |
| device, .to(device) | 모델과 배치를 CPU 또는 GPU에 올리기 |
| class SmallCNN(nn.Module) | 모델 설계도와 파라미터 등록. Conv2d가 옆 픽셀을 보는 층 |
| forward / network(x) | 현재 파라미터로 예측 계산 |
| CrossEntropyLoss | 점수 10개와 정답 번호의 차이를 숫자로 표현 |
| zero_grad | 이전 기울기 기록 비우기 |
| backward | 현재 손실의 기울기 계산 |
| optimizer.step | 기울기를 이용해 파라미터 갱신 |
| main | 이 작업들의 실행 순서 묶기 |

**GPU를 쓰는 이유:** 6번 끝의 CPU·GPU 시간 비교를 보세요. 사진 한 장이 3,072개 숫자이고 5만 장을 여러 바퀴 돌리면 CPU로는 수업 시간 안에 끝나지 않습니다. 3주차 표(240행 × 3열)에서는 이 차이가 보이지 않았습니다.

**직접 바꾸기:** `batch_size=128`을 256으로 바꾸고 3번부터 실행하세요. 한 바퀴의 배치 수와 첫 배치 모양이 어떻게 바뀌나요? 정답: 196배치, 입력 `(256, 3, 32, 32)`, 정답 `(256,)`이고 마지막은 80장입니다. 시험 점수로 여러 설정을 고르는 과제는 아닙니다.

## 선택. RNN 문서에서 입력 모양 읽기

**이하 구역은 선택입니다. 학습된 CNN·RNN 성능을 비교하는 과제가 아닙니다.** CNN은 사진처럼 옆 픽셀 관계가 있는 데이터, RNN은 센서 기록처럼 시간 순서가 있는 데이터에 맞는 구조입니다. 사진은 순서 데이터가 아니므로 여기서는 작은 순서 데이터를 직접 만들어 RNN이 입력을 어떻게 읽는지 모양만 확인합니다.

[RNN 공식 문서](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html): `batch_first=True`이면 입력은 `(N, L, F)`입니다. N은 기록 묶음 수, L은 시간 길이, F는 시점마다의 특징 수입니다. 아래는 **공정 두 번을 각각 네 시점에 걸쳐, 센서 두 개로 측정했다고 가정한 합성 예시**입니다.

`output[:, -1, :]`는 모든 묶음에서 마지막 시점의 은닉 특징을 꺼냅니다. 마지막 시점의 결과 하나를 예측하는 작업을 가정한 선택입니다. 매 시점 예측이 목표라면 출력 선택과 정답 모양도 달라집니다. `batch_first`는 여기서 입력·output의 축 순서를 정하며 hidden의 모양은 여전히 `(층 수, N, 은닉 수)`입니다.

In [ ]:
sequence = torch.tensor([  # 시간 기록 두 묶음을 텐서로 만듭니다.
    [[0.0, 0.1], [0.2, 0.2], [0.4, 0.3], [0.6, 0.4]],  # 첫 공정의 네 시점·두 센서입니다.
    [[0.6, 0.4], [0.4, 0.3], [0.2, 0.2], [0.0, 0.1]],  # 둘째 공정의 네 시점·두 센서입니다.
])  # 전체 모양은 (2, 4, 2)입니다.
rnn = nn.RNN(input_size=2, hidden_size=4, batch_first=True)  # 시점마다 센서 2개를 읽습니다.
sequence_output, last_hidden = rnn(sequence)  # 모든 시점의 특징과 마지막 은닉 상태입니다.
last_feature = sequence_output[:, -1, :]  # 마지막 시점의 특징 4개입니다.
rnn_head = nn.Linear(4, 1)  # 마지막 특징으로 결과 하나를 만듭니다.
rnn_output = rnn_head(last_feature)  # 아직 학습하지 않은 출력입니다.
print("입력:", sequence.shape, "시점별 출력:", sequence_output.shape)  # (2, 4, 2), (2, 4, 4)입니다.
print("마지막 은닉:", last_hidden.shape, "마지막 특징:", last_feature.shape)  # (1, 2, 4), (2, 4)입니다.
print("최종 출력:", rnn_output.shape)  # 공정 두 번에 결과 하나씩, (2, 1)입니다.
print("미학습 출력:", rnn_output.detach())  # 학습하지 않아 의미 있는 예측값은 아닙니다.

In [ ]:
plt.figure(figsize=(6, 4))  # 첫 공정 묶음의 센서 기록입니다.
plt.plot(range(4), sequence[0, :, 0], marker="o", label="Sensor 1")  # 첫 센서의 시간 순서입니다.
plt.plot(range(4), sequence[0, :, 1], marker="o", label="Sensor 2")  # 둘째 센서의 시간 순서입니다.
plt.xlabel("Time step")  # 네 시점입니다.
plt.ylabel("Synthetic sensor value")  # 합성 숫자입니다.
plt.legend()  # 센서를 구분합니다.
plt.tight_layout()  # 여백을 맞춥니다.
plt.show()  # 시간 축의 의미를 확인합니다.

## 다른 아키텍처를 연결할 때 생각할 내용

1. 한 샘플의 뜻과 축 이름부터 씁니다. 표의 행인지, 영상 한 장인지, 시간 묶음인지 구별합니다.
2. 공식 문서의 Shape에서 입력 축 순서와 채널·특징 수를 확인합니다. 필요할 때만 `unsqueeze`, `permute`, `reshape`를 사용하고 전후 `shape`를 출력합니다. `reshape`는 숫자를 재배열할 뿐 축의 의미를 만들어 주지 않습니다.
3. 중간 출력 크기로 다음 층의 입력 수를 정합니다. CNN은 채널·공간 크기, RNN은 시간·은닉 크기를 확인합니다. 4번의 `Linear(64 * 8 * 8, 10)`이 그 예입니다.
4. 무엇을 예측하는지에 맞춰 마지막 층과 정답·손실을 함께 정합니다. 여기서는 10가지 분류와 CrossEntropyLoss를 사용했습니다. 회귀는 출력 하나와 MSE입니다.
5. 영상·시간 자료는 같은 웨이퍼나 로트가 학습과 시험에 섞이지 않도록 분리하고, 시간 문제는 미래 정보가 들어가지 않게 나눕니다.

### 공식 참고 문서

- [CIFAR-10 데이터셋](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html)
- [Dataset / DataLoader](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)
- [nn.Module](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)
- [학습 반복문과 optimizer](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)
- [Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
- [RNN](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html)

기본 API 예제이며 제공된 실행 출력의 torch 버전은 첫 셀에서 확인합니다. 다른 버전의 문서에서는 선택 인자나 기본값이 달라질 수 있으므로 설치한 버전의 문서도 확인합니다.